# Standard Knowledge Distillation — Qwen2.5-0.5B-Instruct student / Qwen2.5-1.5B-Instruct teacher (TRL backend)

In [ ]:
from aligntune.core.backend_factory import create_distill_trainer

# backend="unsloth": this is the "standard distillation" case where student
# and teacher normally share an architecture family (as here). Fixed two
# real bugs in aligntune/backends/unsloth/distill/distillation/distillation.py
# to make this work:
# - The teacher now loads via Unsloth too when it shares the student's
#   architecture family (auto-detected via AutoConfig().model_type, no
#   weights loaded just to check) - loading the student via Unsloth first
#   monkey-patches that architecture's attention/norm classes process-wide,
#   so a plain-HF-loaded same-family teacher inherited those patches without
#   the internal buffers Unsloth's own loader sets up, crashing with
#   "'Qwen2Attention' object has no attribute 'apply_qkv'". Override with
#   teacher_use_unsloth=True/False if you need to force it either way.
# - DistillationConfig's own max_length (default 1024, governs how long TRL's
#   collator lets an example get) was decoupled from max_seq_length (what's
#   passed to FastLanguageModel.from_pretrained) - any example landing
#   between the two sailed through the collator untouched, then got
#   silently truncated by Unsloth inside the model forward, desyncing the
#   logits length from the labels/completion_tokens length
#   DistillationTrainer computed from the untruncated input ("Size does not
#   match at dimension 1" deep inside its loss computation). Now kept equal
#   to max_seq_length automatically - this affects both backends, but only
#   surfaces as a crash on Unsloth since a plain HF model never silently
#   truncates.
# Verified end-to-end with real train loss reported.
trainer = create_distill_trainer(
    student_model="Qwen/Qwen2.5-0.5B-Instruct",
    teacher_model="Qwen/Qwen2.5-1.5B-Instruct",
    dataset_name="tatsu-lab/alpaca",
    split="train",
    backend="unsloth",
    output_dir="./out_distillation",
    batch_size=1,
    num_epochs=1,
    max_steps=10,
    learning_rate=5e-5,
    temperature=1.0,
    alpha=0.5,
    loss_type="kl",
    max_seq_length=128,
    max_samples=512,
    use_peft=True,
    lora_r=8,
    loggers=["none"],
    seed=42,
    eval_strategy="no",
)

results = trainer.train()
print("Standard distillation training completed.")
print(results)

## Online (on-policy) distillation — `on_policy=True`

Same trainer, `on_policy=True` (forces `lmbda=1.0`): the student generates its own
completions during training instead of using dataset/teacher completions.

Found and fixed two real bugs to make this path work:
- `DistillTrainerBase._get_task_type()` (added 2026-07-25, after the fix below)
  special-cased "fully on-policy" distillation to route through `task_type="ppo"`
  (a prompt-only schema with no `messages` column) - bypassing the
  `task_type="distillation_offline"` fix already made in `data/manager.py` on
  2026-07-09 specifically to avoid this. Crashed with `KeyError: 'messages'` in
  TRL's `_DistillationCollator`. Now always routes to `distillation_offline`,
  whose `messages`-shaped output the collator handles for both on-policy
  (prompt-only last turn) and offline (full prompt+completion) cases.
- Unsloth backend only: Unsloth's patched `model.generate()` forces
  `torch.inference_mode()` internally for speed, so the generated token
  sequences come back as inference tensors - any later autograd op that
  touches them (the student log-probs gather in TRL's on-policy divergence
  loss) crashed with "Inference tensors cannot be saved for backward". Fixed
  by cloning the generate() output before use, mirroring the same workaround
  Unsloth's own RL integration (`unsloth/models/rl.py`) applies for its
  supported trainers (GRPO etc.) - `DistillationTrainer` isn't one of those,
  so it never got the clone.


In [ ]:
from aligntune.core.backend_factory import create_distill_trainer

trainer = create_distill_trainer(
    student_model="Qwen/Qwen2.5-0.5B-Instruct",
    teacher_model="Qwen/Qwen2.5-1.5B-Instruct",
    dataset_name="tatsu-lab/alpaca",
    split="train",
    backend="unsloth",
    output_dir="./out_distillation_online",
    batch_size=1,
    num_epochs=1,
    max_steps=10,
    learning_rate=5e-5,
    temperature=1.0,
    alpha=0.5,
    loss_type="kl",
    on_policy=True,
    max_seq_length=128,
    max_samples=512,
    use_peft=True,
    lora_r=8,
    loggers=["none"],
    seed=42,
    eval_strategy="no",
)

results = trainer.train()
print("Online distillation training completed.")
print(results)